<a href="https://colab.research.google.com/github/aaquibmomin2003/LearnMachineLearning/blob/main/house_price_prediction_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset , DataLoader
import numpy as np
np.random.seed(42)

mock_data = {
    'sq_ft': np.random.randint(1000, 4000, size=500),
    'bedrooms': np.random.randint(1, 6, size=500),
    'house_age': np.random.randint(1, 50, size=500),
    'school_rating': np.random.randint(1, 11, size=500),
    'price': np.random.randint(150000, 700000, size=500)
}
pd.DataFrame(mock_data).to_csv('real_housing_data.csv',index=False)
print("Step 1 Complete: 'real_housing_data.csv' created successfully in your sidebar!")

Step 1 Complete: 'real_housing_data.csv' created successfully in your sidebar!


In [8]:
df = pd.read_csv('real_housing_data.csv')


In [3]:
input_columns = ['sq_ft', 'bedrooms', 'house_age', 'school_rating']
X_data = df[input_columns].values
y_data = df['price'].values

In [4]:
df.shape

(500, 5)

In [5]:
X_tensor = torch.tensor(X_data, dtype=torch.float32)
y_tensor = torch.tensor(y_data, dtype=torch.float32)
print("Python Tensor Conversion")

print(f"Inputs Shape : {X_tensor.shape} (500 houses, 4 features each)")
print(f"Targets Shape: {y_tensor.shape} (500 house prices)")

Python Tensor Conversion
Inputs Shape : torch.Size([500, 4]) (500 houses, 4 features each)
Targets Shape: torch.Size([500]) (500 house prices)


In [6]:
housing_dataset = TensorDataset(X_tensor, y_tensor)
housing_loader = DataLoader(housing_dataset, batch_size=32, shuffle=True)

# Grab the first batch just to prove it works!
first_batch_inputs, first_batch_targets = next(iter(housing_loader))
print("\n--- FIRST MINI-BATCH LOADED SUCCESSFULLY ---")
print(f"Batch Input Tensor Shape: {first_batch_inputs.shape}")


--- FIRST MINI-BATCH LOADED SUCCESSFULLY ---
Batch Input Tensor Shape: torch.Size([32, 4])


In [12]:
df[['price']].shape

(500, 1)

In [13]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# 1. ACCELERATION CHECK
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Engine Running On: {device.upper()}\n")

# 2. READ REAL DATA SPREADSHEET (Pandas Pipeline)
df = pd.read_csv('real_housing_data.csv')

# Extract input features and target prices
X_data = df[['sq_ft', 'bedrooms', 'house_age', 'school_rating']].values
y_data = df[['price']].values

# Convert straight to PyTorch Tensors
X_tensor = torch.tensor(X_data, dtype=torch.float32)
y_tensor = torch.tensor(y_data, dtype=torch.float32)

# Create the automated 32-row conveyor belt loader
housing_dataset = TensorDataset(X_tensor, y_tensor)
housing_loader = DataLoader(housing_dataset, batch_size=32, shuffle=True)

# 3. DEFINE ARCHITECTURE
class RealHousingMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden_layer = nn.Linear(4, 8)  # 4 inputs match our 4 spreadsheet columns
        self.output_layer = nn.Linear(8, 1)  # 1 output matches our single price target
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.output_layer(self.relu(self.hidden_layer(x)))

model = RealHousingMLP().to(device)

# 4. LOSS FUNCTION AND OPTIMIZER
loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01) # Learning rate adjusted for scale

# 5. THE MULTI-BATCH REAL DATA TRAINING LOOP
print("--- STARTING PRODUCTION REAL DATA TRAINING ---")

for epoch in range(1, 21): # Run for 20 epochs
    epoch_loss = 0.0

    for batch_inputs, batch_targets in housing_loader:
        # Slam the 32-row chunk onto the GPU
        batch_inputs = batch_inputs.to(device)
        batch_targets = batch_targets.to(device)

        # Standard structural optimization steps
        optimizer.zero_grad()
        predictions = model(batch_inputs)
        loss = loss_function(predictions, batch_targets)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    # Calculate the average penalty across the dataset
    avg_loss = epoch_loss / len(housing_loader)

    if epoch % 2 == 0 or epoch == 1:
        print(f"Epoch {epoch:02d}/20 | Average Batch Error (MSE): {avg_loss:.2f}")

print("\n--- PIPELINE TRAINING COMPLETE ---")

Engine Running On: CUDA

--- STARTING PRODUCTION REAL DATA TRAINING ---
Epoch 01/20 | Average Batch Error (MSE): 197799947264.00
Epoch 02/20 | Average Batch Error (MSE): 196878868480.00
Epoch 04/20 | Average Batch Error (MSE): 194375604224.00
Epoch 06/20 | Average Batch Error (MSE): 188669913088.00
Epoch 08/20 | Average Batch Error (MSE): 179272449024.00
Epoch 10/20 | Average Batch Error (MSE): 164646454784.00
Epoch 12/20 | Average Batch Error (MSE): 146020972544.00
Epoch 14/20 | Average Batch Error (MSE): 127057239040.00
Epoch 16/20 | Average Batch Error (MSE): 109490063360.00
Epoch 18/20 | Average Batch Error (MSE): 92436114944.00
Epoch 20/20 | Average Batch Error (MSE): 77855193344.00

--- PIPELINE TRAINING COMPLETE ---


In [14]:
import pandas as pd
import torch
from sklearn.preprocessing import MinMaxScaler

# 1. LOAD THE CSV SPREADSHEET AS BEFORE
df = pd.read_csv('real_housing_data.csv')

X_data = df[['sq_ft', 'bedrooms', 'house_age', 'school_rating']].values
y_data = df[['price']].values

# =====================================================================
# NEW STEP: INITIALIZE SCALERS AND COMPRESS THE DATA
# =====================================================================
# We use separate scalers for inputs (X) and targets (y)
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

# Fit the scalers to learn the min/max values and transform the data
X_scaled = scaler_X.fit_transform(X_data)
y_scaled = scaler_y.fit_transform(y_data)

# =====================================================================
# CONVERT THE NEW SCALED DATA INTO PYTORCH TENSORS
# =====================================================================
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y_scaled, dtype=torch.float32)

print("--- RAW DATA COPIED FROM SPREADSHEET ---")
print(f"Original Row 1 Features: {X_data[0]}")
print(f"Original Row 1 Price   : ${y_data[0][0]}")

print("\n--- COMPRESSED SCALED DATA SENT TO PYTORCH ---")
print(f"Scaled Row 1 Features  : {X_tensor[0]}")
print(f"Scaled Row 1 Price     : {y_tensor[0].item():.4f}")

--- RAW DATA COPIED FROM SPREADSHEET ---
Original Row 1 Features: [1860    5   37    2]
Original Row 1 Price   : $672828

--- COMPRESSED SCALED DATA SENT TO PYTORCH ---
Scaled Row 1 Features  : tensor([0.2865, 1.0000, 0.7500, 0.1111])
Scaled Row 1 Price     : 0.9523


In [15]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler

# 1. ACCELERATION CHECK
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Engine Running On: {device.upper()}\n")

# 2. LOAD AND SCALE DATA
df = pd.read_csv('real_housing_data.csv')
X_data = df[['sq_ft', 'bedrooms', 'house_age', 'school_rating']].values
y_data = df[['price']].values

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(X_data)
y_scaled = scaler_y.fit_transform(y_data)

X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y_scaled, dtype=torch.float32)

housing_dataset = TensorDataset(X_tensor, y_tensor)
housing_loader = DataLoader(housing_dataset, batch_size=32, shuffle=True)

# 3. DEFINE ARCHITECTURE
class ScaledHousingMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden_layer = nn.Linear(4, 8)
        self.output_layer = nn.Linear(8, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.output_layer(self.relu(self.hidden_layer(x)))

model = ScaledHousingMLP().to(device)

# 4. LOSS FUNCTION AND OPTIMIZER
loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01) # Clean learning rate

# 5. THE SCALED TRAINING LOOP
print("--- STARTING SCALED DATA TRAINING ---")

for epoch in range(1, 21):
    epoch_loss = 0.0

    for batch_inputs, batch_targets in housing_loader:
        batch_inputs = batch_inputs.to(device)
        batch_targets = batch_targets.to(device)

        optimizer.zero_grad()
        predictions = model(batch_inputs)
        loss = loss_function(predictions, batch_targets)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(housing_loader)

    if epoch % 2 == 0 or epoch == 1:
        print(f"Epoch {epoch:02d}/20 | Average Batch Error (MSE): {avg_loss:.4f}")

print("\n--- PIPELINE TRAINING COMPLETE ---")

Engine Running On: CUDA

--- STARTING SCALED DATA TRAINING ---
Epoch 01/20 | Average Batch Error (MSE): 0.0925
Epoch 02/20 | Average Batch Error (MSE): 0.0877
Epoch 04/20 | Average Batch Error (MSE): 0.0871
Epoch 06/20 | Average Batch Error (MSE): 0.0869
Epoch 08/20 | Average Batch Error (MSE): 0.0886
Epoch 10/20 | Average Batch Error (MSE): 0.0865
Epoch 12/20 | Average Batch Error (MSE): 0.0875
Epoch 14/20 | Average Batch Error (MSE): 0.0868
Epoch 16/20 | Average Batch Error (MSE): 0.0884
Epoch 18/20 | Average Batch Error (MSE): 0.0891
Epoch 20/20 | Average Batch Error (MSE): 0.0869

--- PIPELINE TRAINING COMPLETE ---


In [16]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler

device = "cuda" if torch.cuda.is_available() else "cpu"

# (Data loading and scaling remains exactly the same)
df = pd.read_csv('real_housing_data.csv')
X_data = df[['sq_ft', 'bedrooms', 'house_age', 'school_rating']].values
y_data = df[['price']].values

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()
X_scaled = scaler_X.fit_transform(X_data)
y_scaled = scaler_y.fit_transform(y_data)

X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y_scaled, dtype=torch.float32)

housing_dataset = TensorDataset(X_tensor, y_tensor)
housing_loader = DataLoader(housing_dataset, batch_size=32, shuffle=True)

# =====================================================================
# UPGRADED DEEPER & WIDER ARCHITECTURE
# =====================================================================
class DeepHousingMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(4, 64)   # Expanded to 64 neurons
        self.layer2 = nn.Linear(64, 32)  # Added a brand new 32-neuron layer
        self.output = nn.Linear(32, 1)   # Final output guess
        self.relu = nn.ReLU()            # Activation function

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        return self.output(x)

model = DeepHousingMLP().to(device)

# We keep the optimizer and loss function exactly the same to keep the test fair
loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

print("--- STARTING DEEPER NETWORK TRAINING ---")
for epoch in range(1, 21):
    epoch_loss = 0.0
    for batch_inputs, batch_targets in housing_loader:
        batch_inputs = batch_inputs.to(device)
        batch_targets = batch_targets.to(device)

        optimizer.zero_grad()
        predictions = model(batch_inputs)
        loss = loss_function(predictions, batch_targets)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(housing_loader)
    if epoch % 2 == 0 or epoch == 1:
        print(f"Epoch {epoch:02d}/20 | Average Batch Error (MSE): {avg_loss:.4f}")

--- STARTING DEEPER NETWORK TRAINING ---
Epoch 01/20 | Average Batch Error (MSE): 0.1010
Epoch 02/20 | Average Batch Error (MSE): 0.0886
Epoch 04/20 | Average Batch Error (MSE): 0.0868
Epoch 06/20 | Average Batch Error (MSE): 0.0874
Epoch 08/20 | Average Batch Error (MSE): 0.0874
Epoch 10/20 | Average Batch Error (MSE): 0.0902
Epoch 12/20 | Average Batch Error (MSE): 0.0861
Epoch 14/20 | Average Batch Error (MSE): 0.0882
Epoch 16/20 | Average Batch Error (MSE): 0.0890
Epoch 18/20 | Average Batch Error (MSE): 0.0885
Epoch 20/20 | Average Batch Error (MSE): 0.0891


In [17]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler

device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. GENERATE LOGICAL DATA (With a real mathematical pattern)
np.random.seed(42)
sq_ft = np.random.randint(1000, 4000, size=500)
bedrooms = np.random.randint(1, 6, size=500)
house_age = np.random.randint(1, 50, size=500)
school_rating = np.random.randint(1, 11, size=500)

# Price is now directly calculated from the features!
price = (sq_ft * 150) + (bedrooms * 10000) - (house_age * 2000) + (school_rating * 5000)

mock_data = {
    'sq_ft': sq_ft, 'bedrooms': bedrooms, 'house_age': house_age,
    'school_rating': school_rating, 'price': price
}
pd.DataFrame(mock_data).to_csv('real_housing_data.csv', index=False)

# 2. LOAD AND SCALE
df = pd.read_csv('real_housing_data.csv')
X_data = df[['sq_ft', 'bedrooms', 'house_age', 'school_rating']].values
y_data = df[['price']].values

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()
X_scaled = scaler_X.fit_transform(X_data)
y_scaled = scaler_y.fit_transform(y_data)

X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y_scaled, dtype=torch.float32)

housing_dataset = TensorDataset(X_tensor, y_tensor)
housing_loader = DataLoader(housing_dataset, batch_size=32, shuffle=True)

# 3. DEEP ARCHITECTURE
class PatternHousingMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(4, 64)
        self.layer2 = nn.Linear(64, 32)
        self.output = nn.Linear(32, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        return self.output(x)

model = PatternHousingMLP().to(device)
loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

print("--- TRAINING ON LOGICAL PATTERN DATA ---")
for epoch in range(1, 21):
    epoch_loss = 0.0
    for batch_inputs, batch_targets in housing_loader:
        batch_inputs = batch_inputs.to(device)
        batch_targets = batch_targets.to(device)

        optimizer.zero_grad()
        predictions = model(batch_inputs)
        loss = loss_function(predictions, batch_targets)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(housing_loader)
    if epoch % 2 == 0 or epoch == 1:
        print(f"Epoch {epoch:02d}/20 | Average Batch Error (MSE): {avg_loss:.6f}")

--- TRAINING ON LOGICAL PATTERN DATA ---
Epoch 01/20 | Average Batch Error (MSE): 0.056001
Epoch 02/20 | Average Batch Error (MSE): 0.004063
Epoch 04/20 | Average Batch Error (MSE): 0.000237
Epoch 06/20 | Average Batch Error (MSE): 0.000058
Epoch 08/20 | Average Batch Error (MSE): 0.000038
Epoch 10/20 | Average Batch Error (MSE): 0.000031
Epoch 12/20 | Average Batch Error (MSE): 0.000027
Epoch 14/20 | Average Batch Error (MSE): 0.000021
Epoch 16/20 | Average Batch Error (MSE): 0.000019
Epoch 18/20 | Average Batch Error (MSE): 0.000015
Epoch 20/20 | Average Batch Error (MSE): 0.000014


In [18]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# 1. LOAD THE PATTERN SPREADSHEET
df = pd.read_csv('real_housing_data.csv')
X_data = df[['sq_ft', 'bedrooms', 'house_age', 'school_rating']].values
y_data = df[['price']].values

# =====================================================================
# NEW STEP: SPLIT INTO TRAIN (80%) AND TEST (20%) SECTIONS FIRST
# =====================================================================
# test_size=0.20 means 20% goes to the final exam. random_state keeps the shuffle consistent.
X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, test_size=0.20, random_state=42)

# =====================================================================
# SCALE THE SEPARATE PILES
# =====================================================================
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

# CRITICAL: We only let the scalers "learn" (fit) from the Training pile!
X_train_scaled = scaler_X.fit_transform(X_train)
y_train_scaled = scaler_y.fit_transform(y_train)

# Then we just transform the test pile using those same boundaries
X_test_scaled = scaler_X.transform(X_test)
y_test_scaled = scaler_y.transform(y_test)

print("--- DATA SEPARATION SECTOR COMPLETE ---")
print(f"Total Houses in Dataset : {len(df)}")
print(f"Houses for Training (80%): {X_train_scaled.shape[0]}")
print(f"Houses for Testing (20%) : {X_test_scaled.shape[0]}")

--- DATA SEPARATION SECTOR COMPLETE ---
Total Houses in Dataset : 500
Houses for Training (80%): 400
Houses for Testing (20%) : 100


In [19]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# 1. ACCELERATION CHECK
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Engine Running On: {device.upper()}\n")

# 2. LOAD AND SPLIT DATA (80% Train, 20% Test)
df = pd.read_csv('real_housing_data.csv')
X_data = df[['sq_ft', 'bedrooms', 'house_age', 'school_rating']].values
y_data = df[['price']].values

X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, test_size=0.20, random_state=42)

# Scale the data split piles separately
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
y_train_scaled = scaler_y.fit_transform(y_train)
X_test_scaled = scaler_X.transform(X_test)
y_test_scaled = scaler_y.transform(y_test)

# Convert both piles into PyTorch Tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32).to(device) # Send entire test set to GPU
y_test_tensor = torch.tensor(y_test_scaled, dtype=torch.float32).to(device)

# We only build a batch loader for the TRAINING data
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# 3. ARCHITECTURE
class ProductionHousingMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(4, 64)
        self.layer2 = nn.Linear(64, 32)
        self.output = nn.Linear(32, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        return self.output(x)

model = ProductionHousingMLP().to(device)
loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# 4. THE TRAIN-TEST VALIDATION LOOP
print("--- STARTING PRODUCTION VALIDATION LOOP ---")

for epoch in range(1, 21):
    # --- PHASE 1: TRAINING ---
    model.train() # Set model to training mode
    train_epoch_loss = 0.0

    for batch_inputs, batch_targets in train_loader:
        batch_inputs = batch_inputs.to(device)
        batch_targets = batch_targets.to(device)

        optimizer.zero_grad()
        predictions = model(batch_inputs)
        loss = loss_function(predictions, batch_targets)
        loss.backward()
        optimizer.step()

        train_epoch_loss += loss.item()

    avg_train_loss = train_epoch_loss / len(train_loader)

    # --- PHASE 2: TESTING (THE FINAL EXAM) ---
    model.eval() # Freeze layers for evaluation
    with torch.no_grad(): # Freeze weight adjustments
        test_predictions = model(X_test_tensor)
        test_loss = loss_function(test_predictions, y_test_tensor).item()

    if epoch % 2 == 0 or epoch == 1:
        print(f"Epoch {epoch:02d}/20 | Train Loss: {avg_train_loss:.6f} | Unseen Test Loss: {test_loss:.6f}")

print("\n--- PERFORMANCE EVALUATION COMPLETE ---")

Engine Running On: CUDA

--- STARTING PRODUCTION VALIDATION LOOP ---
Epoch 01/20 | Train Loss: 0.034176 | Unseen Test Loss: 0.006068
Epoch 02/20 | Train Loss: 0.002904 | Unseen Test Loss: 0.001677
Epoch 04/20 | Train Loss: 0.000531 | Unseen Test Loss: 0.000281
Epoch 06/20 | Train Loss: 0.000192 | Unseen Test Loss: 0.000127
Epoch 08/20 | Train Loss: 0.000093 | Unseen Test Loss: 0.000054
Epoch 10/20 | Train Loss: 0.000054 | Unseen Test Loss: 0.000053
Epoch 12/20 | Train Loss: 0.000043 | Unseen Test Loss: 0.000047
Epoch 14/20 | Train Loss: 0.000036 | Unseen Test Loss: 0.000038
Epoch 16/20 | Train Loss: 0.000031 | Unseen Test Loss: 0.000034
Epoch 18/20 | Train Loss: 0.000024 | Unseen Test Loss: 0.000023
Epoch 20/20 | Train Loss: 0.000031 | Unseen Test Loss: 0.000036

--- PERFORMANCE EVALUATION COMPLETE ---


In [20]:
import numpy as np

# A user logs onto your site and inputs data for a single house:
# Specs: 2,500 sq_ft, 3 bedrooms, 15 years old, school_rating of 8
new_house_specs = np.array([[2500, 3, 15, 8]])

# 1. COMPRESS: We must scale the inputs exactly like we scaled the training data
new_house_scaled = scaler_X.transform(new_house_specs)
new_house_tensor = torch.tensor(new_house_scaled, dtype=torch.float32).to(device)

# 2. PREDICT: Pass the scaled tensor through the frozen model
model.eval()
with torch.no_grad():
    scaled_prediction = model(new_house_tensor)

# 3. DECOMPRESS: Turn the decimal output back into actual dollar currency
# .cpu().numpy() brings the tensor back from the T4 GPU back to standard Python memory
real_dollar_prediction = scaler_y.inverse_transform(scaled_prediction.cpu().numpy())

print("--- REAL-WORLD INFERENCE RESULT ---")
print(f"House Characteristics: {new_house_specs[0][0]} sq ft, {new_house_specs[0][1]} beds, {new_house_specs[0][2]} years old")
print(f"AI Appraiser Valuation: ${real_dollar_prediction[0][0]:,.2f}")

--- REAL-WORLD INFERENCE RESULT ---
House Characteristics: 2500 sq ft, 3 beds, 15 years old
AI Appraiser Valuation: $412,838.72
